## Création du client pour le workspace

In [1]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

# authenticate
credential = DefaultAzureCredential()
SUBSCRIPTION ="f5c48a87-8c0c-4d6f-8113-52045b4c6355"
RESOURCE_GROUP ="stephenpetieu-rg"
WS_NAME ="Mastering_AI"

# Get a handle to the workspace
ml_client = MLClient(
    credential=credential,
    subscription_id=SUBSCRIPTION,
    resource_group_name=RESOURCE_GROUP,
    workspace_name=WS_NAME,
)

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


## Vérification de l'existence du modèle

In [2]:
registered_model_name = "credit_defaults_model"

# Let's pick the latest version of the model
latest_model_version = max(
    [int(m.version) for m in ml_client.models.list(name=registered_model_name)]
)

print(latest_model_version)

4


## Création de l'Endpoint

In [3]:
from azure.ai.ml.entities import ManagedOnlineEndpoint
import uuid

# Create a unique name for the endpoint
online_endpoint_name = "credit-endpoint-" + str(uuid.uuid4())[:8]

# define an online endpoint
endpoint = ManagedOnlineEndpoint(
    name=online_endpoint_name,
    description="this is an online endpoint",
    auth_mode="key",
    tags={
        "training_dataset": "credit_defaults",
    },
)
# create the online endpoint
# expect the endpoint to take approximately 2 minutes.

endpoint = ml_client.online_endpoints.begin_create_or_update(endpoint).result()

/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/mlflow/__init__.py:41: UserWarning: Versions of mlflow (3.8.1) and child packages mlflow-skinny (3.5.0) are different. This may lead to unexpected behavior. Please install the same version of all MLflow packages.
  mlflow.mismatch._check_version_mismatch()


#### Vérification du statut de l'Endpoint

In [4]:
endpoint = ml_client.online_endpoints.get(name=online_endpoint_name)

print(
    f'Endpoint "{endpoint.name}" with provisioning state "{endpoint.provisioning_state}" is retrieved'
)

Endpoint "credit-endpoint-2409edc5" with provisioning state "Succeeded" is retrieved


## Création du déploiement

In [5]:
from azure.ai.ml.entities import ManagedOnlineDeployment

# Choose the latest version of the registered model for deployment
model = ml_client.models.get(name=registered_model_name, version=latest_model_version)

# define an online deployment
# if you run into an out of quota error, change the instance_type to a comparable VM that is available.
# Learn more on https://azure.microsoft.com/en-us/pricing/details/machine-learning/.
blue_deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name=online_endpoint_name,
    model=model,
    environment="azureml:credit_defaults_model:2",
    instance_type="Standard_DS3_v2",
    instance_count=1,
    environment_variables={
        "AZUREML_COLLECT_MODEL_DATA": "false",
        "MLFLOW_MODEL_FOLDER": "credit_defaults_model",  # <-- nom du dossier MLflow dans l'artefact
    },
)
# create the online deployment
blue_deployment = ml_client.online_deployments.begin_create_or_update(
    blue_deployment
).result()

# blue deployment takes 100% traffic
# expect the deployment to take approximately 8 to 10 minutes.
endpoint.traffic = {"blue": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()

Check: endpoint credit-endpoint-2409edc5 exists
Readonly attribute principal_id will be ignored in class <class 'azure.ai.ml._restclient.v2022_05_01.models._models_py3.ManagedServiceIdentity'>
Readonly attribute tenant_id will be ignored in class <class 'azure.ai.ml._restclient.v2022_05_01.models._models_py3.ManagedServiceIdentity'>


...............................................................................

ManagedOnlineEndpoint({'public_network_access': 'Enabled', 'provisioning_state': 'Succeeded', 'scoring_uri': 'https://credit-endpoint-2409edc5.francecentral.inference.ml.azure.com/score', 'openapi_uri': 'https://credit-endpoint-2409edc5.francecentral.inference.ml.azure.com/swagger.json', 'name': 'credit-endpoint-2409edc5', 'description': 'this is an online endpoint', 'tags': {'training_dataset': 'credit_defaults'}, 'properties': {'createdBy': 'Stephen NTANTAME PETIEU', 'createdAt': '2026-03-18T12:38:28.160423+0000', 'lastModifiedAt': '2026-03-18T12:52:50.944199+0000', 'azureml.onlineendpointid': '/subscriptions/f5c48a87-8c0c-4d6f-8113-52045b4c6355/resourcegroups/stephenpetieu-rg/providers/microsoft.machinelearningservices/workspaces/mastering_ai/onlineendpoints/credit-endpoint-2409edc5', 'AzureAsyncOperationUri': 'https://management.azure.com/subscriptions/f5c48a87-8c0c-4d6f-8113-52045b4c6355/providers/Microsoft.MachineLearningServices/locations/francecentral/mfeOperationsStatus/oeidp:

#### Vérification du status, et de quelques méta données

In [6]:
# return an object that contains metadata for the endpoint
endpoint = ml_client.online_endpoints.get(name=online_endpoint_name)

# print a selection of the endpoint's metadata
print(
    f"Name: {endpoint.name}\nStatus: {endpoint.provisioning_state}\nDescription: {endpoint.description}"
)

Name: credit-endpoint-2409edc5
Status: Succeeded
Description: this is an online endpoint


In [8]:
# existing traffic details
print(endpoint.traffic)

# Get the scoring URI
print(endpoint.scoring_uri)

{'blue': 100}
https://credit-endpoint-2409edc5.francecentral.inference.ml.azure.com/score


## Test du Enpoint avec des données de test.

#### Enregistrement des données de test dans un JSON

In [9]:
import os

# Create a directory to store the sample request file.
deploy_dir = "./deploy"
os.makedirs(deploy_dir, exist_ok=True)

In [10]:
%%writefile {deploy_dir}/sample-request.json
{
  "input_data": {
    "columns": [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22],
    "index": [0, 1],
    "data": [
            [20000,2,2,1,24,2,2,-1,-1,-2,-2,3913,3102,689,0,0,0,0,689,0,0,0,0],
            [10, 9, 8, 7, 6, 5, 4, 3, 2, 1, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1, 10, 9, 8]
            ]
  }
}

Overwriting ./deploy/sample-request.json


#### Envoie des données de test vers le endpoint et affichage des logs

In [11]:
# test the blue deployment with the sample data
ml_client.online_endpoints.invoke(
    endpoint_name=online_endpoint_name,
    deployment_name="blue",
    request_file="./deploy/sample-request.json",
)

'[1, 0]'

In [12]:
logs = ml_client.online_deployments.get_logs(
    name="blue", endpoint_name=online_endpoint_name, lines=50
)
print(logs)

Instance status:
SystemSetup: Succeeded
UserContainerImagePull: Succeeded
ModelDownload: Succeeded
UserContainerStart: Succeeded

Container events:
Kind: Pod, Name: Downloading, Type: Normal, Time: 2026-03-18T12:50:23.141105Z, Message: Start downloading models
Kind: Pod, Name: Pulling, Type: Normal, Time: 2026-03-18T12:50:23.681972Z, Message: Start pulling container image
Kind: Pod, Name: Pulled, Type: Normal, Time: 2026-03-18T12:51:41.228232Z, Message: Container image is pulled successfully
Kind: Pod, Name: Downloaded, Type: Normal, Time: 2026-03-18T12:51:41.228232Z, Message: Models are downloaded successfully
Kind: Pod, Name: Created, Type: Normal, Time: 2026-03-18T12:51:41.264779Z, Message: Created container inference-server
Kind: Pod, Name: Started, Type: Normal, Time: 2026-03-18T12:51:41.744076Z, Message: Started container inference-server
Kind: Pod, Name: ContainerReady, Type: Normal, Time: 2026-03-18T12:52:03.745269383Z, Message: Container is ready

Container logs:
2026-03-18 12

## Création d'un second endpoint

In [13]:
green_deployment = ManagedOnlineDeployment(
    name="green",
    endpoint_name=online_endpoint_name,
    model=model,
    environment="azureml:credit_defaults_model:2",
    instance_type="Standard_E4s_v3",
    instance_count=1,
    environment_variables={
        "MLFLOW_MODEL_FOLDER": "credit_defaults_model",  # <-- nom du dossier MLflow dans l'artefact
    },
)
# create the online deployment
green_deployment = ml_client.online_deployments.begin_create_or_update(
    green_deployment
).result()

Check: endpoint credit-endpoint-2409edc5 exists


........................................................................

#### Scale-up du déploiement pour prendre en main plus de traffic

In [14]:
# update definition of the deployment
blue_deployment.instance_count = 2

# update the deployment
# expect the deployment to take approximately 8 to 10 minutes
ml_client.online_deployments.begin_create_or_update(blue_deployment).result()

Check: endpoint credit-endpoint-2409edc5 exists


HttpResponseError: (BadRequest) The request is invalid.
Code: BadRequest
Message: The request is invalid.
Exception Details:	(InferencingClientCallFailed) {"error":{"code":"Validation","message":"{\"errors\":{\"ScaleSettings\":[\"Not enough quota available for Standard_DS3_v2 in SubscriptionId f5c48a87-8c0c-4d6f-8113-52045b4c6355. Current usage/limit: 12/14. Additional needed: 4 Please see troubleshooting guide, available here: https://aka.ms/oe-tsg#error-outofquota\"]},\"type\":\"https://tools.ietf.org/html/rfc9110#section-15.5.1\",\"title\":\"One or more validation errors occurred.\",\"status\":400,\"traceId\":\"00-1ba7f16674b90d4f483b4dcb3c77ab39-79c8964a6eb3095b-01\"}"}}
	Code: InferencingClientCallFailed
	Message: {"error":{"code":"Validation","message":"{\"errors\":{\"ScaleSettings\":[\"Not enough quota available for Standard_DS3_v2 in SubscriptionId f5c48a87-8c0c-4d6f-8113-52045b4c6355. Current usage/limit: 12/14. Additional needed: 4 Please see troubleshooting guide, available 

#### Mise à jour de l'allocation des traffics

In [15]:
endpoint.traffic = {"blue": 20, "green": 80}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()

Readonly attribute principal_id will be ignored in class <class 'azure.ai.ml._restclient.v2022_05_01.models._models_py3.ManagedServiceIdentity'>
Readonly attribute tenant_id will be ignored in class <class 'azure.ai.ml._restclient.v2022_05_01.models._models_py3.ManagedServiceIdentity'>


ManagedOnlineEndpoint({'public_network_access': 'Enabled', 'provisioning_state': 'Succeeded', 'scoring_uri': 'https://credit-endpoint-2409edc5.francecentral.inference.ml.azure.com/score', 'openapi_uri': 'https://credit-endpoint-2409edc5.francecentral.inference.ml.azure.com/swagger.json', 'name': 'credit-endpoint-2409edc5', 'description': 'this is an online endpoint', 'tags': {'training_dataset': 'credit_defaults'}, 'properties': {'createdBy': 'Stephen NTANTAME PETIEU', 'createdAt': '2026-03-18T12:38:28.160423+0000', 'lastModifiedAt': '2026-03-18T13:20:02.578181+0000', 'azureml.onlineendpointid': '/subscriptions/f5c48a87-8c0c-4d6f-8113-52045b4c6355/resourcegroups/stephenpetieu-rg/providers/microsoft.machinelearningservices/workspaces/mastering_ai/onlineendpoints/credit-endpoint-2409edc5', 'AzureAsyncOperationUri': 'https://management.azure.com/subscriptions/f5c48a87-8c0c-4d6f-8113-52045b4c6355/providers/Microsoft.MachineLearningServices/locations/francecentral/mfeOperationsStatus/oeidp:

#### Test avec plusieurs appels et affichage des logs

In [16]:
# You can invoke the endpoint several times
for i in range(30):
    ml_client.online_endpoints.invoke(
        endpoint_name=online_endpoint_name,
        request_file="./deploy/sample-request.json",
    )
logs = ml_client.online_deployments.get_logs(
    name="green", endpoint_name=online_endpoint_name, lines=50
)
print(logs)

#### Envoyer tous les traffic vers le deuxième déploiement

In [ ]:
endpoint.traffic = {"blue": 0, "green": 100}
ml_client.begin_create_or_update(endpoint).result()